# Project 04 · Customer segmentation for a SACCO

**Benson M. Merita · MeritaAnalytics** · Tools: Python (pandas, scikit-learn), Matplotlib

**Business question.** Which natural groups of members exist, and what should the SACCO do differently for each?

**Data.** 650 members described by seven behaviour measures: transactions per month, average transaction (Ksh), average balance (Ksh), number of loans, tenure in months, share of transactions done through an agent, and app logins. Synthetic, generated with a fixed seed. The file also contains the final segment labels; they are dropped at the start so the analysis rebuilds them from scratch.

| Step | What it answers |
|---|---|
| 1 | Look at the data |
| 2 | Prepare the measures (log and scale) |
| 3 | How many groups? |
| 4 | Are the groups stable? |
| 5 | See the groups on a map (PCA) |
| 6 | Profile and name each group |
| 7 | One action and one metric per group |

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, GREEN, GREY, DARK, AMBER = "#1F5FD6", "#2BB38A", "#9CA3AF", "#0B2A4A", "#B45309"
plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)

cust = pd.read_csv("../datasets/04-customer-segments/customers_segmented.csv", index_col="customer_id")
cust = cust.drop(columns=["segment_id", "segment"])     # remove the published answer; rebuild it below
print(cust.shape)

## Step 1 · Look at the data

**What:** summary statistics per measure.
**Why:** money and count columns are usually *skewed*: most members have small values and a few have very large ones. That affects the next step.

In [ ]:
cust.describe().loc[["mean", "50%", "max"]].round(1).T

## Step 2 · Prepare the measures

**What:** apply a log transform to the skewed columns, then **standardise** every column to mean 0 and standard deviation 1.
**Why:** k-means groups members by distance. Without scaling, a column in shillings (thousands) would swamp a column like app logins (tens). The log stops a handful of very rich members from deciding the groups.

In [ ]:
from sklearn.preprocessing import StandardScaler

skewed = ["txns_per_month", "avg_txn_ksh", "avg_balance_ksh", "app_logins", "loan_count"]
feat = cust.copy()
feat[skewed] = np.log1p(feat[skewed])
Z = StandardScaler().fit_transform(feat)
print("skew before:", cust[skewed].skew().round(2).to_dict())
print("skew after :", feat[skewed].skew().round(2).to_dict())

## Step 3 · How many groups?

**What:** run k-means for 2 to 8 groups and compute the **silhouette score** for each (from −1 to 1; higher means members sit clearly closer to their own group than to others). As a baseline, score four *random* groups.
**Why:** k-means will always return groups, even from noise. The baseline proves the structure is real.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

random_labels = np.random.default_rng(0).integers(0, 4, len(Z))
print("baseline, 4 random groups, silhouette:", round(silhouette_score(Z, random_labels), 3))

ks = list(range(2, 9))
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    inertia.append(km.inertia_); sil.append(silhouette_score(Z, km.labels_))
pd.DataFrame({"k": ks, "inertia": np.round(inertia, 0), "silhouette": np.round(sil, 3)})

In [ ]:
fig, ax = plt.subplots(1, 2)
ax[0].plot(ks, inertia, marker="o", color=BLUE); ax[0].set_title("Elbow plot")
ax[0].set_xlabel("Number of groups k"); ax[0].set_ylabel("Within-group spread (inertia)")
ax[1].plot(ks, sil, marker="o", color=GREEN); ax[1].set_title("Silhouette by k")
ax[1].set_xlabel("Number of groups k"); ax[1].set_ylabel("Mean silhouette")
plt.tight_layout(); plt.show()

Four groups give the highest silhouette, far above the random baseline, and the elbow bends at the same point. We use **k = 4**.

## Step 4 · Are the groups stable?

**What:** refit with five other random starts, and once on a bootstrap sample (members drawn with replacement). Compare with the **adjusted Rand index (ARI)**: 1.0 means identical grouping.
**Why:** groups that change every time the model runs cannot be used to plan campaigns.

In [ ]:
from sklearn.metrics import adjusted_rand_score

K = ks[int(np.argmax(sil))]
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Z)
cust["segment_id"] = km.labels_
print("k used:", K, "| group sizes:", cust.segment_id.value_counts().sort_index().to_dict())

aris = [adjusted_rand_score(km.labels_, KMeans(K, n_init=10, random_state=s).fit(Z).labels_) for s in range(1, 6)]
print("ARI against other random starts, lowest:", round(min(aris), 3))
idx = np.random.default_rng(1).choice(len(Z), len(Z))
km_boot = KMeans(K, n_init=10, random_state=42).fit(Z[idx])
print("ARI, bootstrap model applied to all members:", round(adjusted_rand_score(km.labels_, km_boot.predict(Z)), 3))

## Step 5 · See the groups on a map

**What:** PCA squeezes the seven measures into two axes that keep as much of the variation as possible, so the groups can be drawn.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42).fit(Z)
P2, ev = pca.transform(Z), pca.explained_variance_ratio_
print("variation shown on the map:", round(ev.sum(), 3))
palette = [BLUE, GREEN, AMBER, DARK]
fig, ax = plt.subplots()
for s in range(K):
    m = cust.segment_id.to_numpy() == s
    ax.scatter(P2[m, 0], P2[m, 1], s=12, color=palette[s], alpha=0.7, label=f"group {s}")
ax.set_xlabel(f"PC1 ({ev[0]:.0%} of variation)"); ax.set_ylabel(f"PC2 ({ev[1]:.0%} of variation)")
ax.set_title("Members on a 2-D map, coloured by group"); ax.legend(ncol=K)
plt.tight_layout(); plt.show()

## Step 6 · Profile and name each group

**What:** the typical (median) member of each group, in shillings and counts. Names come from a simple rule: the group with the highest balance is "Savers", the highest transactions "Power users", the highest agent share "Agent-reliant", and the remaining group "Light users".
**Why:** a manager acts on "Savers hold Ksh 89,000", not on "cluster 2".

In [ ]:
prof = cust.groupby("segment_id").agg(
    members=("txns_per_month", "size"), txns=("txns_per_month", "median"),
    balance=("avg_balance_ksh", "median"), loans=("loan_count", "mean"),
    tenure=("tenure_months", "median"), agent=("agent_share", "mean"), logins=("app_logins", "median"))
prof["share_pct"] = (100 * prof.members / prof.members.sum()).round(1)

names, left = {}, set(prof.index)
for label, col in [("Savers", "balance"), ("Power users", "txns"), ("Agent-reliant", "agent")]:
    pick = prof.loc[list(left), col].idxmax(); names[pick] = label; left.discard(pick)
for s in left:
    names[s] = "Light users"
prof["segment"] = prof.index.map(names)
cust["segment"] = cust.segment_id.map(names)
prof.set_index("segment").round(2)

## Step 7 · One action and one metric per group

In [ ]:
actions = {
    "Savers": ("Offer fixed-deposit or goal-savings products; consider loans against savings", "balance and savings uptake"),
    "Power users": ("Protect with fast support and fee tiers; cross-sell merchant accounts", "transactions per month, churn"),
    "Agent-reliant": ("Agent-assisted onboarding to the app; keep agent float healthy", "app logins, agent share"),
    "Light users": ("First-loan offer or airtime bonus for a second transaction", "30-day reactivation rate"),
}
plan = pd.DataFrame([(n, int((cust.segment == n).sum()), *actions[n]) for n in prof.segment],
                    columns=["segment", "members", "action", "metric to track"])
plan

## Key findings and recommendations

- **Four groups** separate clearly (silhouette about 0.47 against −0.03 for random groups) and are **fully stable** (ARI 1.0).
- **Savers** hold by far the largest balances but rarely transact: the natural audience for deposit products and loans secured on savings.
- **Agent-reliant** members barely use the app. Moving them to it cuts agent costs, but only with assisted onboarding.
- **Light users** are the largest and newest group, so their first months decide whether they stay.
- Treat each action as a test: try it on a sample of members before rolling it out.

**Limitations.** Synthetic members separate far more cleanly than real ones. Expect lower silhouette scores on real data and review the groups every quarter.